# 01 · MSHMM catalog

Each row in `db.xlsx` represents one satellite observation of a site. A site may have observations from multiple dates and sensors.\
This notebook introduces the catalog fields and basic queries. Distributions, maps, and imagery are covered in later notebooks.

#### 1. Set paths and read the catalog

`DATA_ROOT` points to the folder containing `db.xlsx`, `Batch0`, `Batch1`, and `Batch2`.\
Change `DATA_ROOT` with your local data path.

In [ ]:
from pathlib import Path
import sys

from IPython.display import display

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebook":
    REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT / "src"))

from mshmm.catalog import load_catalog, describe_fields

DATA_ROOT = REPO_ROOT.parent  # By default, data is stored in the parent directory of the script. Change this to your data root.
catalog = load_catalog(DATA_ROOT)

print(f"Observations: {len(catalog):,}, fields: {len(catalog.columns)}")
display(catalog.head(3))

Observations: 1,686, fields: 16


,belonging,name,lon,lat,type,sensor,time,path,batch,is_plume,Q,Q_err,IME,length,wind,wind_direction
0,America,DCP Midstream oil grid #4,-103.748,32.219,oil,GF5,2022-05-09 01:43:40+00:00,Batch1\output\GF5B_AHSI_W103.7_N32.3_20220508_...,1,True,4235.33,447.09,296.51,"[907.26, 1016.83, 1151.36, 898.78, 600.0, 635....",6.90,54.81
1,America,DCP Midstream oil grid #4,-103.748,32.219,oil,GF5,2023-12-05 01:49:38+00:00,Batch1\output\GF5B_AHSI_W103.5_N32.3_20231204_...,1,True,2758.49,535.54,389.25,"[1043.1, 981.43, 669.4, 659.17]",3.09,-19.52
2,America,DCP Midstream oil grid #4,-103.748,32.219,oil,PRS,2024-10-07 17:42:57+00:00,Batch2\output\PRS_L1_STD_OFFL_20241007174257_2...,2,False,NaN,NaN,NaN,NaN,NaN,NaN


#### 2. Fields of catalog

The catalog contains 16 fields, detailed as follows:

In [11]:
display(describe_fields(catalog))

,meaning,unit,notes,dtype,missing
belonging,Country or region,—,Country or country-admin region,object,0
name,Site name,—,Name of the emission source site,object,0
lon,Longitude,°,WGS84,float64,0
lat,Latitude,°,WGS84,float64,0
type,Source category,—,waste / oil / coal,object,0
sensor,Satellite sensor,—,EMIT / ENMAP / PRS (PRISMA) / GF5,object,0
time,Acquisition time,UTC,Parsed as a timezone-aware pandas timestamp,"datetime64[ns, UTC]",0
path,Record directory,—,Relative to the data root containing db.xlsx,object,0
batch,Data batch,—,0 / 1 / 2,int64,0
is_plume,Plume annotation status,—,True / False; used to select annotated plume r...,bool,0


#### 3. Emission events and Non-emission events
The emission event records correspond to rows where `is_plume=true` , and all fields should be valid.\
Non-emission event records correspond to rows where `is_plume=false`, and have missing values in `Q`, `Q_err`, `IME`, `length`, `wind`, and `wind_direction`. 


In [8]:
quantification_fields = ["Q", "Q_err", "IME", "length", "wind", "wind_direction"]

# Compare quantification fields for one plume and one non-plume record.
plume_example = catalog.loc[catalog["is_plume"]].head(1)
non_plume_example = catalog.loc[~catalog["is_plume"]].head(1)
columns = ["name", "sensor", "is_plume", *quantification_fields]
display(plume_example[columns])
display(non_plume_example[columns])

,name,sensor,is_plume,Q,Q_err,IME,length,wind,wind_direction
0,DCP Midstream oil grid #4,GF5,True,4235.33,447.09,296.51,"[907.26, 1016.83, 1151.36, 898.78, 600.0, 635....",6.9,54.81


,name,sensor,is_plume,Q,Q_err,IME,length,wind,wind_direction
2,DCP Midstream oil grid #4,PRS,False,NaN,NaN,NaN,NaN,NaN,NaN


#### 4. Filter records

Combine pandas boolean conditions to select records by sensor, source category, and annotation status.\
Here we select EMIT observations of landfill sites with annotated plumes.

In [9]:
selected = catalog.loc[
    (catalog["sensor"] == "EMIT")
    & (catalog["type"] == "waste")
    & catalog["is_plume"]
]

print(f"Selected records: {len(selected)}")
print("Original row indices: ", selected.index[:5].tolist())
display(selected[["name", "sensor", "type", "time", "Q"]].head())

Selected records: 102
Original row indices:  [143, 144, 145, 146, 147]


,name,sensor,type,time,Q
143,Simeprodeso,EMIT,waste,2023-01-30 18:49:23+00:00,1182.23
144,Simeprodeso,EMIT,waste,2023-03-25 21:25:07+00:00,3432.78
145,Simeprodeso,EMIT,waste,2023-04-02 18:15:02+00:00,5563.81
146,Simeprodeso,EMIT,waste,2023-06-23 18:49:06+00:00,1248.24
147,Simeprodeso,EMIT,waste,2023-09-24 20:57:57+00:00,1430.19


#### 5. Locate a data record

The `path` field is relative to `DATA_ROOT`.\
To avoid system differences, Replace Windows10-style backslashes with forward slashes before joining paths with `Path`.

In [10]:
record = selected.iloc[0]
record_dir = DATA_ROOT / record["path"].replace("\\", "/")

print("Original row index: ", record.name)
print("Record directory: ", record_dir.relative_to(DATA_ROOT))
print("Directory exists: ", record_dir.is_dir())

Original row index:  143
Record directory:  Batch0\data\Hyper_TROPOMI_landfill\Mexico\Nuevo_Leon\Simeprodeso\EMIT_L1B_RAD_001_20230130T184923_2303013_006_plume0\-100.3_25.87
Directory exists:  True
